<a href="https://colab.research.google.com/github/Chanaporn042/GE338-Lab-2/blob/main/6606614672.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install earthengine-api geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 18.4 MB/s eta 0:00:00


In [ ]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='ee-chanaporn040')

In [ ]:
import ee
import geemap

# =======================
# 1. กำหนดพื้นที่
# =======================
gaul = ee.FeatureCollection("FAO/GAUL/2015/level1")
phichit = gaul.filter(ee.Filter.eq('ADM1_NAME', 'Phichit'))

Map = geemap.Map(center=[16.4, 100.3], zoom=9)
Map.addLayer(phichit, {}, 'Phichit Boundary')


# =======================
# 2. ฟังก์ชัน mask เมฆ
# =======================
def maskS2clouds(image):
    scl = image.select('SCL')

    mask = (scl.eq(4)
            .Or(scl.eq(5))
            .Or(scl.eq(6))
            .Or(scl.eq(7)))

    return image.updateMask(mask).divide(10000)


# =======================
# 3. โหลด Sentinel-2
# =======================
def getComposite(start, end):
    collection = (ee.ImageCollection("COPERNICUS/S2_SR")
        .filterBounds(phichit)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
        .map(maskS2clouds)
        .select(['B3', 'B4', 'B8', 'B11']))

    return collection.median().clip(phichit)


s2_2020 = getComposite('2020-01-01', '2020-04-30')
s2_2024 = getComposite('2024-01-01', '2024-04-30')


# =======================
# 4. คำนวณ Index
# =======================

# NDVI
ndvi_2020 = s2_2020.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndvi_2024 = s2_2024.normalizedDifference(['B8', 'B4']).rename('NDVI')

# NDWI
ndwi_2020 = s2_2020.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndwi_2024 = s2_2024.normalizedDifference(['B3', 'B8']).rename('NDWI')

# NDMI
ndmi_2020 = s2_2020.normalizedDifference(['B8', 'B11']).rename('NDMI')
ndmi_2024 = s2_2024.normalizedDifference(['B8', 'B11']).rename('NDMI')


# =======================
# 5. Change detection
# =======================
ndvi_change = ndvi_2024.subtract(ndvi_2020)
ndmi_change = ndmi_2024.subtract(ndmi_2020)


# =======================
# 6. Visualization
# =======================
ndviVis = {'min': 0, 'max': 1, 'palette': ['brown', 'yellow', 'green']}
ndwiVis = {'min': -1, 'max': 1, 'palette': ['brown', 'white', 'blue']}
changeVis = {'min': -0.5, 'max': 0.5, 'palette': ['red', 'white', 'green']}

Map.addLayer(ndvi_2020, ndviVis, 'NDVI 2020')
Map.addLayer(ndvi_2024, ndviVis, 'NDVI 2024')
Map.addLayer(ndwi_2024, ndwiVis, 'NDWI 2024')
Map.addLayer(ndmi_2024, ndviVis, 'NDMI 2024')

Map.addLayer(ndvi_change, changeVis, 'NDVI Change')
Map.addLayer(ndmi_change, changeVis, 'NDMI Change')

Map

Map(center=[16.4, 100.3], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

In [ ]:
# 5. Export NDMI ไป Google Drive
# ======================================
task = ee.batch.Export.image.toDrive(
    image=ndvi_change,
    description='Phichit_NDVI_change',
    folder='Lab2',        # Folder บน Google Drive
    fileNamePrefix='NDVI_change', # ชื่อไฟล์
    region=phichit.geometry(),
    scale=20,                   # 🔹 ใช้ 20m ลด memory
    crs='EPSG:4326',
    maxPixels=1e13
)

task.start()
print("Export started! ตรวจสอบสถานะได้ด้วย task.status()")

Export started! ตรวจสอบสถานะได้ด้วย task.status()
